# 12. Scopes & Closures
Exhaustive guide to scopes, lookup rules (LEGB), nonlocal/global variables, stateful closures, scope leakages and namespaces collections.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for combined analysis questions at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Local vs Global Scope Namespaces
**Explanation**: Variables map directly to local or global scopes.

**Syntax**:
```python
global_variable = value
def function_name():
    local_variable = value
```



In [ ]:
global_namespace_value = 1
def print_values():
    local_namespace_value = 2
    print('g:', global_namespace_value, 'l:', local_namespace_value)
print_values()

### 2. Enclosing Scopes
**Explanation**: Nested functions can read variables from enclosing scopes.

**Syntax**:
```python
def outer_function():
    enclosing_variable = value
    def inner_function(): print(enclosing_variable)
```



In [ ]:
def outer_scope_fn():
    enclosing_scope_value = 10
    def inner_scope_fn(): print('Enclosing:', enclosing_scope_value)
    inner_scope_fn()
outer_scope_fn()

### 3. Built-in Namespaces
**Explanation**: Contain pre-defined Python methods and properties.

**Syntax**:
```python
len(iterable_object)
```



In [ ]:
import builtins
print('Builtin id:', id(builtins.len))

### 4. LEGB Lookup Sequence
**Explanation**: Resolves scopes in order: Local -> Enclosing -> Global -> Built-in.

**Syntax**:
```python
# Resolves variable reference
```

**Visual Explanation (Data with Baraa Style)**:
```mermaid
graph TD
    L[Local Scope] --> E[Enclosing Scope] --> G[Global Scope] --> B[Built-in Scope]
```


In [ ]:
global_scope_variable = 'Global'
def outer_function():
    def inner_function(): print(global_scope_variable)
    inner_function()
outer_function()

### 5. Modifying Global Variables (global)
**Explanation**: Modifies global variables within local function frames.

**Syntax**:
```python
global global_variable_name
```



In [ ]:
global_counter_variable = 1
def increment_global():
    global global_counter_variable
    global_counter_variable = 2
increment_global()
print(global_counter_variable)

### 6. Modifying Enclosing Variables (nonlocal)
**Explanation**: Modifies enclosing variables within nested scope blocks.

**Syntax**:
```python
nonlocal enclosing_variable_name
```



In [ ]:
def outer_counter_fn():
    enclosing_counter = 1
    def inner_increment():
        nonlocal enclosing_counter
        enclosing_counter = 2
    inner_increment()
    print(enclosing_counter)
outer_counter_fn()

### 7. Stateful Closures
**Explanation**: Closures capture enclosing state bindings even after parent scope exits.

**Syntax**:
```python
def parent_function(state_value):
    def inner_function(): return state_value
```

**Visual Explanation (Data with Baraa Style)**:
```mermaid
graph TD
    outer[outer function frame exits] -.-> closure([Closure Object retains 'multiplier_value'])
```


In [ ]:
def make_multiplier_closure(multiplier_value):
    return lambda: multiplier_value
multiplier_instance = make_multiplier_closure(10)
print(multiplier_instance())

### 8. Accessing Closure Cell Variables
**Explanation**: Inspect free variables stored inside closures.

**Syntax**:
```python
closure_function.__closure__
```



In [ ]:
def make_state_closure(state_value):
    return lambda: state_value
state_instance = make_state_closure('closure_val')
print(state_instance.__closure__[0].cell_contents)

### 9. Multiple Independent Closure Instances
**Explanation**: Each closure call instantiates isolated namespaces.

**Syntax**:
```python
instance_one = outer(value_one)
instance_two = outer(value_two)
```



In [ ]:
first_instance = make_state_closure(1)
second_instance = make_state_closure(2)
print('Isolated closure namespaces?:', first_instance() != second_instance())

### 10. Nested Scope Levels Scoping Trees
**Explanation**: Nested lookups traverse nested scopes sequentially.

**Syntax**:
```python
def level_one():
    def level_two():
        def level_three(): pass
```



In [ ]:
def outer_nested_fn():
    nested_level_value = 1
    def middle_nested_fn():
        def inner_nested_fn(): print(nested_level_value)
        inner_nested_fn()
    middle_nested_fn()
outer_nested_fn()

### 11. Dynamic Variable Binding Resolution
**Explanation**: Closures lookup variables at call execution time, not definition time.

**Syntax**:
```python
# Late bindings resolution
```



In [ ]:
dynamic_lookup_variable = 5
def print_lookup_value(): print(dynamic_lookup_variable)
dynamic_lookup_variable = 10
print_lookup_value()

### 12. Scopes in Loop Variables Leakage
**Explanation**: For loop indices leak into the enclosing module namespace.

**Syntax**:
```python
for index_variable in range(3): pass
print(index_variable) # leaks index_variable
```



In [ ]:
for loop_leak_index in range(3): pass
print('Leaked loop index k:', loop_leak_index)

### 13. Scopes in Comprehensions Namespace Isolation
**Explanation**: Comprehension loop index variables do not leak into the parent scope.

**Syntax**:
```python
[comprehension_variable for comprehension_variable in range(3)]
```



In [ ]:
try:
    [comprehension_leak_index for comprehension_leak_index in range(3)]
    print(comprehension_leak_index)
except NameError as error_message:
    print('Comprehension variable did not leak:', error_message)

### 14. Function attribute namespaces
**Explanation**: Functions can store custom properties directly in their attribute namespaces.

**Syntax**:
```python
function_name.attribute_name = value
```



In [ ]:
def dummy_function(): pass
dummy_function.execution_counter = 100
print('Function attribute counter:', dummy_function.execution_counter)

### 15. Reading namespace dict collections
**Explanation**: Exposes globals variables state dictionary copy.

**Syntax**:
```python
globals().copy()
```



In [ ]:
print('Is csv_path in globals?:', 'csv_path' in globals())

## Section 3: Fintech Interview Questions

### Q1: Implement stateful closure logger tracking running transaction amount sums by card_type, logging from the first 10 rows.

In [ ]:
# Solution:
def make_card_tracker():
    totals = {}
    def tracker(card, amount):
        nonlocal totals
        totals[card] = totals.get(card, 0.0) + amount
        return totals
    return tracker

tracker = make_card_tracker()
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(10):
        row = f.readline().strip().split(',')
        card = row[4].strip()
        amount = float(row[3]) if row[3] not in ('', 'NaN') else 0.0
        print('Totals:', tracker(card, amount))


### Q2: Use function attribute variables to track transaction parser execution call counts on the first 5 rows.

In [ ]:
# Solution:
def parse_row(row):
    parse_row.calls = getattr(parse_row, 'calls', 0) + 1
    return row[0]

with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(5):
        row = f.readline().strip().split(',')
        parse_row(row)
print('Calls count logged on function namespace:', parse_row.calls)
